In [ ]:

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
#imports
from pyspark.sql.functions import current_timestamp, col
from awsglue.dynamicframe import DynamicFrame 
from datetime import datetime
#s3 bucket connections
bucket = "bitcoindatapipelineproject"
raw_path = f"s3://{bucket}/raw_layer/unprocessed_raw/"
output_path = f"s3://{bucket}/refined_layer/"
#read in our data
raw_df = (spark.read.option("multiLine", True).json(raw_path))
#raw_df.show()
#raw_df.count()
#transformation functions, renaming cols, cleaning data
def metadata(df):
    meta = (
        df.select(
            col("id").alias("coin_id"),
            col("symbol"),
            col("name"),
            col("image").alias("image_url"),
            col("market_cap_rank"),
            col("max_supply")
        )
        .dropDuplicates(["coin_id"])
        .withColumn("extracted_at", current_timestamp())
        
    )
    return meta 

def marketdata(df):
    market = (
        df.select(
            col("id").alias("coin_id"),
            col("current_price").alias("price_usd"),
            col("market_cap").alias("market_cap_usd"),
            col("total_volume").alias("volume_usd"),
            col("high_24h").alias("high_24h_usd"),
            col("low_24h").alias("low_24h_usd"),
            col("price_change_24h"),
            col("price_change_percentage_24h").alias("price_change_pct_24h"),
            col("market_cap_change_24h"),
            col("market_cap_change_percentage_24h").alias("market_cap_change_pct_24h")
        )
        .dropDuplicates(["coin_id"])
        .withColumn("extracted_at", current_timestamp())
    )
    return market
        
#dataframe creation
metadata_df = metadata(raw_df)
marketdata_df = marketdata(raw_df)
#metadata_df.head()
#marketdata_df.head()
#export data to s3 bucket
def write_to_s3(df,path,fmt = 'csv'):
    
    #df = df.coalesce(1) #create 1 file output only
    dyf = DynamicFrame.fromDF(df, glueContext, 'dyf') #enhanced dataframe with added Glue functions
    glueContext.write_dynamic_frame.from_options(
        frame = dyf,
        connection_type = "s3",
        connection_options = {"path": path},
        format = fmt,
        format_options = {"header": True}
    )
    
    
#write our data to s3
write_to_s3(marketdata_df,"s3://bitcoindatapipelineproject/refined_layer/market_data/market_data_transformed_{}".format(datetime.now().strftime("%Y-%m-%d")),"csv")
write_to_s3(metadata_df, "s3://bitcoindatapipelineproject/refined_layer/meta_data/meta_data_transformed_{}".format(datetime.now().strftime("%Y-%m-%d")), "csv")
job.commit()

In [ ]:
import boto3
def get_raw_filenames_s3(bucket, prefix):
    crypto_keys = []
    s3 = boto3.client("s3")
    for file in s3.list_objects(Bucket=bucket, Prefix=prefix)['Contents']:
        file_name = file['Key']
        if file_name.split('.')[-1] == "json":
            crypto_keys.append(file_name)
    return crypto_keys
bucket = bucket
prefix = "raw_layer/unprocessed_raw/"
raw_unprocessed_crypto_files = get_raw_filenames_s3(bucket, prefix)
raw_unprocessed_crypto_files
def copy_delete_raw(bucket,crypto_keys):
    s3_resource = boto3.resource('s3')
    for key in crypto_keys:
        source_location = {
            'Bucket': bucket,
            'Key': key
        }
        s3_resource.meta.client.copy(source_location, bucket, 'raw_layer/processed_raw/' + key.split("/")[-1])    
        s3_resource.Object(bucket, key).delete()
copy_delete_raw(bucket,raw_unprocessed_crypto_files)
job.commit()